# Konoros-Sec v1.1 — Colab Train & Test (defensive only)
Repo đã có sẵn data thật: data/raw/prime_all.jsonl (280 Q&A elite sau dedup). Máy 83GB RAM: chọn GPU mạnh nhất có thể, epochs đã set sẵn (tiny 3, small 2). Data càng nhiều thì epochs càng giá trị.

In [ ]:
!pip -q install torch numpy pyyaml tqdm tensorboard pytest requests datasets
!python -m pytest test/ -q  # kỳ vọng: pass hết (gồm test GQA + KV-cache)

In [ ]:
# 1. Đổ thêm data mới (bulk, cần SE_API_KEY để lên 10k req/ngày) + build .bin
import os
os.environ["SE_API_KEY"] = "PASTE_KEY_HERE"  # bỏ dòng này nếu dùng quota ẩn danh
!python data_prime/stackoverflow_elite/api_prime.py --out data_prime/raw/so_elite.jsonl --max 2000 --tags python,javascript,linux,security,networking --pages 5 --pagesize 30
!python data_prime/wiki_elite/hf_stream.py --max-en 8000 --max-vi 4000
!python data_prime/build/merge_prime.py --out data/raw/prime_all.jsonl
!python scripts/prepare_data.py --input data/raw/prime_all.jsonl,data/raw/train.txt --train-out data/processed/train.bin --val-out data/processed/val.bin --tok-out data/tokenizer/byte.json

In [ ]:
# 2a. Pretrain TINY (epochs=3, batch 64)
!python -m training.train --config config/model/tiny.yaml
# 2b. Pretrain SMALL (~30M, GQA 8q/2kv, ctx 1024, epochs=2) — chạy khi tiny đã loss giảm
# !python -m training.train --config config/model/small.yaml

In [ ]:
!ls experiments/v0.1/checkpoints/
!python -m evaluation.evaluate --config config/model/tiny.yaml --ckpt experiments/v0.1/checkpoints/step_005000.pt

In [ ]:
# 3. SFT v1.1 defensive
!python -m training.sft --config config/model/tiny.yaml --data data/sft/security_sft.jsonl --base-ckpt experiments/v0.1/checkpoints/step_005000.pt --out experiments/v1.1/sft.pt --max-steps 60 --lr 1e-5

In [ ]:
!python -m security.eval.safety_eval --config config/model/tiny.yaml --ckpt experiments/v1.1/sft.pt
!python -m inference.generate --ckpt experiments/v1.1/sft.pt --prompt "<user>How do I secure my home WiFi?</user>" --max-new-tokens 80 --temperature 0.0

In [ ]:
# 3c. Branched generation: sample 4 nhánh, tự chấm (format + ít lặp), trả nhánh tốt nhất
# !python -m inference.generate --ckpt experiments/v1.1/sft.pt --prompt "<user>How do I secure my home WiFi?</user>" --max-new-tokens 80 --temperature 0.8 --top-p 0.9 --best-of 4

In [ ]:
# 3b. DPO preference (v1.2 alignment): học chosen > rejected. Chạy sau SFT, trước GRPO.
# !python -m training.dpo --config config/model/tiny.yaml --data data/sft/security_prefs.jsonl --base-ckpt experiments/v1.1/sft.pt --out experiments/v1.2/dpo.pt --steps 40
# !python -m security.eval.pref_eval --config config/model/tiny.yaml --ckpt experiments/v1.2/dpo.pt

In [ ]:
# 4. RL suy nghĩ nhiều bước (GRPO): SFT reasoning trước rồi RL. Chạy sau khi SFT xong.
# SFT warmup với think traces:
# !python -m training.sft --config config/model/tiny.yaml --data data/sft/security_reasoning.jsonl --base-ckpt experiments/v1.1/sft.pt --out experiments/v1.3/sft_reason.pt --max-steps 60 --lr 1e-5
# (khuyến nghị: base từ DPO experiments/v1.2/dpo.pt để có pipeline SFT → DPO → GRPO)
# GRPO (G=4, ~vài phút trên T4 với tiny):
# !python -m training.grpo --config config/model/tiny.yaml --data data/sft/security_reasoning.jsonl --base-ckpt experiments/v1.3/sft_reason.pt --out experiments/v1.3/grpo.pt --steps 50 --G 4
# !python -m security.eval.reasoning_eval --config config/model/tiny.yaml --ckpt experiments/v1.3/grpo.pt

### Dump full GĐ2 (khi cần scale lên GB, máy RAM lớn)
```
!apt -qq install -y p7zip-full
!pip -q install py7zr lxml
!wget -O /tmp/so_posts.7z https://archive.org/download/stackexchange/stackoverflow.com-Posts.7z
!python data_prime/stackoverflow_elite/dump_prime.py --in /tmp/so_posts.7z --out data_prime/raw/so_dump_elite.jsonl --max 200000
```